# 01. SurvFace training + 공식 test ArcFace 임베딩 추출

선택된 `training_manifest.csv`의 development/calibration을 먼저 추출한 뒤 공식 gallery → mated probe → unmated probe 순서로 ArcFace 512D 임베딩을 DB에 저장합니다. Training development는 PCA/PQ fit 전용이고 공식 test는 평가 전용입니다. SurvFace는 이미 face-crop 데이터셋이므로 완료된 RGB uint8 NHWC 112×112 aligned bundle을 Grad-CAM과 동일한 검증·등록 PyTorch ArcFace checkpoint에 직접 배치 입력하며 얼굴 검출기를 다시 실행하지 않습니다.

| 범위 | 예상 시간(현재 장비 기준) |
| --- | ---: |
| `EXECUTE_STAGE=False` | 1초 미만 |
| `MODE=dev`, 작은 `DATA_FRACTION` | 선택 비율에 따라 수시간 |
| `MODE=real`, `DATA_FRACTION=1.0` | 20~30시간 이상 |

> **진행/체크포인트/재시작**: batch마다 DB commit과 ledger flush를 수행하되 로그는 약 10% 경계에서만 출력합니다. 중단되면 Kernel Restart 후 처음부터 실행하면 이미 저장된 동일 run/vector 행은 건너뛰고 다음 행부터 진행합니다. full run에서 입력·추론·DB 실패가 하나라도 남으면 phase를 완료 처리하지 않습니다. 전처리 또는 모델 정책을 바꾸면 같은 run에 섞지 말고 00부터 새 run을 만드십시오.


In [ ]:
# Step 1 실행 범위: 이 셀의 세 값만 바꾸고 Kernel Restart -> Run All
MODE = 'real'            # 'dev' 또는 'real'
DATA_FRACTION = 1.0     # 0 < DATA_FRACTION <= 1
SEED = 42

import sys
from pathlib import Path

for _scope_root in (Path.cwd(), *Path.cwd().parents):
    if (_scope_root / 'research').is_dir():
        break
else:
    raise FileNotFoundError('D:/ronbun 내부에서 노트북을 실행하십시오.')
if str(_scope_root) not in sys.path:
    sys.path.insert(0, str(_scope_root))

from research.compression import PCA_SWEEP_DIMENSIONS
from research.experiments.scope import ExperimentScope

PCA_DIMENSIONS = (384, 256, 128, 64, 32)
PQ_SOURCE_DIMENSION = 512
if PCA_DIMENSIONS != tuple(PCA_SWEEP_DIMENSIONS):
    raise RuntimeError('노트북 PCA sweep과 공통 압축 정의가 다릅니다.')
EXPERIMENT_SCOPE = ExperimentScope(
    mode=MODE, data_fraction=DATA_FRACTION, seed=SEED
)
EXPERIMENT_SCOPE.as_dict()


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path


def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("D:/ronbun 내부에서 노트북을 실행하십시오.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from research.runtime import ProgressReporter, RunStore, resolve_active_run

EXECUTE_STAGE = True
RUN_ROOT = PROJECT_ROOT / "runs" / "survface"

def resolve_run_for_preflight():
    try:
        return resolve_active_run(
            RUN_ROOT, environment_variable="RONBUN_SURVFACE_RUN_DIR"
        ), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"

RUN_DIR, RUN_RESOLUTION_ERROR = resolve_run_for_preflight()
LIMIT = None  # 비율은 DATA_FRACTION으로 제어; 디버깅 외에는 행 단위 제한 금지
BATCH_SIZE = 64
USE_CUDA = os.environ.get("RONBUN_USE_CUDA", "1") == "1"
FULL_RUN_ACKNOWLEDGEMENT = "SURVFACE_FULL_EXTRACTION"
FAIL_ON_ANY_MISSING_FOR_OFFICIAL = True
PROGRESS = ProgressReporter("SurvFace 01 ArcFace extraction", heartbeat_seconds=None, milestone_percent=10)


## 1. 실행 안전장치

행 단위 `LIMIT`는 역할과 identity 비율을 깨뜨릴 수 있으므로 기본값은 `None`입니다. 범위는 준비 단계의 identity-aware `DATA_FRACTION`으로 줄입니다. `real, 1.0`만 확인 문자열이 필요합니다.


In [ ]:
preflight = {
    "execute_stage": EXECUTE_STAGE,
    "run_dir": str(RUN_DIR) if RUN_DIR else None,
    "run_resolution_error": RUN_RESOLUTION_ERROR,
    "limit": LIMIT,
    "batch_size": BATCH_SIZE,
    "cuda_requested": USE_CUDA,
    "full_run_acknowledged": FULL_RUN_ACKNOWLEDGEMENT == "SURVFACE_FULL_EXTRACTION",
    "scope": EXPERIMENT_SCOPE.as_dict(),
}
preflight


## 2. batch 추출 및 DB 체크포인트

`origin_512`만 원본 검색 벡터로 저장합니다. 원본 파일은 image lineage에 사용하고, 모델 입력은 `aligned_faces.npy`의 RGB 112×112 crop입니다. DB 비밀번호는 `RONBUN_DB_PASSWORD` 또는 git 제외된 `configs/database.local.yaml`에서 읽습니다.


In [ ]:
result = {"status": "not_executed", **preflight}
if EXECUTE_STAGE:
    if RUN_DIR is None:
        raise RuntimeError(f"SurvFace run을 찾지 못했습니다: {RUN_RESOLUTION_ERROR}")
    if EXPERIMENT_SCOPE.is_paper_run and FULL_RUN_ACKNOWLEDGEMENT != "SURVFACE_FULL_EXTRACTION":
        raise RuntimeError("전체 실행 전 FULL_RUN_ACKNOWLEDGEMENT를 정확히 입력하십시오.")

    import csv

    import numpy as np
    import pandas as pd

    from research.compression import ORIGIN_512
    from research.database import (
        VectorRepository,
        create_database_engine,
        init_database,
        load_database_settings,
        session_scope,
    )
    from research.embeddings import (
        create_pytorch_adapter_from_spec,
        read_model_spec,
    )
    from research.runtime.hashing import sha256_file
    from research.runtime.redaction import redact

    run = RunStore.open(RUN_DIR)
    run.verify_inputs()
    run.verify_phase_artifacts("00_official_protocol_and_run_freeze")
    embedding_config = run.config["embedding"]
    if int(embedding_config["batch_size"]) != BATCH_SIZE:
        raise ValueError("동결된 embedding.batch_size와 노트북 BATCH_SIZE가 다릅니다.")
    if (
        embedding_config.get("framework") != "pytorch"
        or embedding_config.get("source_color_order") != "rgb"
        or list(embedding_config.get("input_size", [])) != [112, 112]
        or embedding_config.get("device") != "cuda"
    ):
        raise ValueError("동결된 ArcFace 입력 계약은 RGB 112×112여야 합니다.")
    if bool(embedding_config.get("detector_inference", True)):
        raise ValueError("SurvFace embedding은 detector_inference=false여야 합니다.")
    data_dir = PROJECT_ROOT / "data" / "interim" / "survface"
    training = pd.read_csv(data_dir / "training_manifest.csv")
    official = pd.read_csv(data_dir / "official_manifest.csv")
    training["protocol_index"] = range(len(training))
    training["official_identity_id"] = pd.NA
    training["source_protocol_index"] = pd.NA
    manifest = pd.concat([training, official], ignore_index=True, sort=False)
    role_order = {
        "training": 0,
        "gallery": 1,
        "registered_probe": 2,
        "unknown_unknown_probe": 3,
    }
    manifest["_role_order"] = manifest["protocol_role"].map(role_order)
    if manifest["_role_order"].isna().any():
        raise ValueError("알 수 없는 protocol_role이 있습니다.")
    manifest = manifest.sort_values(["_role_order", "protocol_index"], kind="stable")
    aligned_dir = (PROJECT_ROOT / run.config["dataset"]["aligned_bundle_dir"]).resolve()
    aligned_manifest_path = aligned_dir / "bundle_manifest.json"
    aligned_manifest = json.loads(aligned_manifest_path.read_text(encoding="utf-8"))
    aligned_index = pd.read_csv(aligned_dir / "aligned_index.csv")
    aligned_faces = np.load(
        aligned_dir / "aligned_faces.npy", mmap_mode="r", allow_pickle=False
    )
    if (
        aligned_manifest.get("preprocessing", {}).get("mode")
        != "official_face_crop_resize"
        or aligned_manifest.get("detector", {}).get("enabled") is not False
        or int(aligned_manifest.get("counts", {}).get("failed", -1)) != 0
        or aligned_faces.shape != (len(manifest), 112, 112, 3)
        or aligned_faces.dtype != np.uint8
        or len(aligned_index) != len(manifest)
    ):
        raise RuntimeError("동결된 SurvFace aligned bundle 계약이 올바르지 않습니다.")
    alignment = aligned_index.loc[:, [
        "sample_id", "aligned_face_index", "source_content_sha256",
        "aligned_content_sha256", "preprocessing_mode",
    ]].rename(columns={"sample_id": "image_id"})
    manifest = manifest.merge(
        alignment, on="image_id", how="left", sort=False, validate="one_to_one"
    )
    if manifest["aligned_face_index"].isna().any():
        raise RuntimeError("manifest와 aligned bundle의 image_id가 일치하지 않습니다.")
    manifest["aligned_face_index"] = manifest["aligned_face_index"].astype(int)
    aligned_indices = manifest["aligned_face_index"].to_numpy(dtype=np.int64)
    if (
        not np.array_equal(np.sort(aligned_indices), np.arange(len(manifest)))
        or not manifest["preprocessing_mode"].eq(
            "official_face_crop_resize"
        ).all()
    ):
        raise RuntimeError("aligned row index 또는 preprocessing mode가 올바르지 않습니다.")
    full_row_count = len(manifest)
    rows = manifest if LIMIT is None else manifest.head(int(LIMIT))

    if not USE_CUDA:
        raise RuntimeError("정식 SurvFace embedding 추출은 CUDA를 요구합니다.")
    model_spec_path = (
        PROJECT_ROOT / embedding_config["model_spec_path"]
    ).resolve()
    model_spec = read_model_spec(model_spec_path, verify_checkpoint=True)
    if model_spec.model_uid != embedding_config["model_uid"]:
        raise ValueError("동결된 model_uid와 ModelSpec이 다릅니다.")
    extractor = create_pytorch_adapter_from_spec(
        model_spec, device=embedding_config["device"]
    )
    if extractor.device.type != "cuda":
        raise RuntimeError("ArcFace adapter가 CUDA 장치에 올라가지 않았습니다.")
    active_device = str(extractor.device)
    aligned_bundle_manifest_sha256 = sha256_file(aligned_manifest_path)
    engine = create_database_engine(load_database_settings())
    init_database(engine)
    counts = {"processed": 0, "inserted": 0, "skipped": 0, "failed": 0}

    with run.phase("01_official_arcface_embedding_extraction") as phase:
        suffix = f"A{phase.attempt:03d}"
        ledger_path = phase.attempt_dir / f"extraction_ledger_{suffix}.csv"
        fields = [
            "protocol_role", "protocol_index", "image_id", "identity_id",
            "official_identity_id", "image_path", "content_sha256", "status",
            "aligned_face_index", "aligned_content_sha256", "preprocessing_mode",
            "embedding_id", "raw_embedding_norm", "error_type", "message",
        ]
        with ledger_path.open("w", encoding="utf-8", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=fields, lineterminator="\n")
            writer.writeheader()
            records = rows.to_dict(orient="records")
            for batch_start in range(0, len(records), BATCH_SIZE):
                batch = records[batch_start:batch_start + BATCH_SIZE]
                batch_ledgers = []
                pending = []
                with session_scope(engine) as session:
                    repository = VectorRepository(session)
                    for row in batch:
                        counts["processed"] += 1
                        image_path = Path(str(row["image_path"]))
                        image_path = image_path if image_path.is_absolute() else PROJECT_ROOT / image_path
                        ledger = {
                            "protocol_role": row["protocol_role"],
                            "protocol_index": int(row["protocol_index"]),
                            "image_id": row["image_id"],
                            "identity_id": row["identity_id"],
                            "official_identity_id": row.get("official_identity_id"),
                            "image_path": str(image_path),
                            "content_sha256": row["source_content_sha256"],
                            "aligned_face_index": int(row["aligned_face_index"]),
                            "aligned_content_sha256": row["aligned_content_sha256"],
                            "preprocessing_mode": row["preprocessing_mode"],
                            "status": None,
                            "embedding_id": None,
                            "raw_embedding_norm": None,
                            "error_type": None,
                            "message": None,
                        }
                        try:
                            if not image_path.is_file():
                                raise FileNotFoundError(image_path)
                            content_sha256 = str(row["source_content_sha256"])
                            if len(content_sha256) != 64:
                                raise ValueError("source_content_sha256가 올바르지 않습니다.")
                            with session.begin_nested():
                                image = repository.add_image(
                                    str(image_path.resolve()),
                                    label=str(row["identity_id"]),
                                    content_sha256=content_sha256,
                                    file_size_bytes=image_path.stat().st_size,
                                )
                                existing = repository.get_embeddings_512(
                                    image_id=image.id,
                                    vector_type=ORIGIN_512,
                                    run_uid=run.run_id,
                                )
                                if existing:
                                    counts["skipped"] += 1
                                    ledger.update(status="skipped_existing", embedding_id=existing[0].id)
                                else:
                                    pending.append((row, image, ledger))
                        except Exception as exc:
                            counts["failed"] += 1
                            failure = redact({
                                "error_type": type(exc).__name__, "message": str(exc)
                            })
                            ledger.update(status="failed", **failure)
                        batch_ledgers.append(ledger)

                    if pending:
                        try:
                            face_indices = np.asarray(
                                [item[0]["aligned_face_index"] for item in pending],
                                dtype=np.int64,
                            )
                            embedding_output = extractor.embed(
                                np.asarray(aligned_faces[face_indices], dtype=np.uint8)
                            )
                            embedding_batch = embedding_output.normalized_embedding
                            raw_norm_batch = embedding_output.raw_norm
                        except Exception as exc:
                            failure = redact({
                                "error_type": type(exc).__name__, "message": str(exc)
                            })
                            counts["failed"] += len(pending)
                            for _, _, ledger in pending:
                                ledger.update(status="failed", **failure)
                        else:
                            for (row, image, ledger), embedding, raw_norm in zip(
                                pending, embedding_batch, raw_norm_batch, strict=True
                            ):
                                try:
                                    with session.begin_nested():
                                        record = repository.add_embedding_512(
                                            image.id,
                                            ORIGIN_512,
                                            {
                                                "run_id": run.run_id,
                                                "model_uid": embedding_output.model_uid,
                                                "checkpoint_sha256": embedding_output.checkpoint_sha256,
                                                "preprocess_hash": embedding_output.preprocess_hash,
                                                "l2_normalized": True,
                                                "dataset": "qmul-survface-v1",
                                                "preprocessing_mode": "official_face_crop_resize",
                                                "aligned_bundle_manifest_sha256": aligned_bundle_manifest_sha256,
                                            },
                                            embedding,
                                            log=json.dumps({
                                                "aligned_face_index": int(row["aligned_face_index"]),
                                                "aligned_content_sha256": row["aligned_content_sha256"],
                                                "raw_embedding_norm": float(raw_norm),
                                                "detector_inference": False,
                                            }),
                                            run_uid=run.run_id,
                                        )
                                    counts["inserted"] += 1
                                    ledger.update(
                                        status="inserted",
                                        embedding_id=record.id,
                                        raw_embedding_norm=float(raw_norm),
                                    )
                                except Exception as exc:
                                    counts["failed"] += 1
                                    failure = redact({
                                        "error_type": type(exc).__name__, "message": str(exc)
                                    })
                                    ledger.update(status="failed", **failure)

                    for ledger in batch_ledgers:
                        writer.writerow(redact(ledger))
                handle.flush()
                PROGRESS.milestone(
                    "batch DB commit/ledger flush 완료",
                    completed=counts["processed"],
                    total=len(rows),
                    inserted=counts["inserted"],
                    skipped=counts["skipped"],
                    failed=counts["failed"],
                )

        full_scope_complete = (
            LIMIT is None
            and counts["processed"] == full_row_count
            and counts["failed"] == 0
        )
        full_protocol_complete = bool(
            EXPERIMENT_SCOPE.is_paper_run and full_scope_complete
        )
        summary = {
            "counts": counts,
            "requested_rows": int(len(rows)),
            "selected_manifest_rows": int(full_row_count),
            "training_manifest_rows": int(len(training)),
            "official_manifest_rows": int(len(official)),
            "execution_scope": EXPERIMENT_SCOPE.as_dict(),
            "full_scope_complete": full_scope_complete,
            "full_protocol_complete": full_protocol_complete,
            "vector_type": ORIGIN_512,
            "framework": "pytorch",
            "active_device": active_device,
            "model_uid": model_spec.model_uid,
            "checkpoint_sha256": model_spec.checkpoint.sha256,
            "preprocess_hash": model_spec.preprocessing.preprocess_hash,
            "aligned_bundle_manifest_sha256": aligned_bundle_manifest_sha256,
            "preprocessing_mode": "official_face_crop_resize",
            "detector_inference": False,
            "model_inference": "frozen_pytorch_embedding_adapter",
        }
        summary_path = phase.attempt_dir / f"extraction_summary_{suffix}.json"
        summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
        if EXPERIMENT_SCOPE.is_paper_run and FAIL_ON_ANY_MISSING_FOR_OFFICIAL and counts["failed"]:
            raise RuntimeError(
                f"공식 coverage에 {counts['failed']}개 실패가 남았습니다. "
                f"checkpoint: {ledger_path}"
            )
        phase.publish_artifact(ledger_path)
        phase.publish_artifact(summary_path)
        phase.record_counts(**counts)
        phase.record("coverage", **summary)
    result = {"status": "completed", "run_id": run.run_id, **summary}
else:
    PROGRESS.emit("검토 모드 완료: 이미지 decode/DB 쓰기를 실행하지 않음", expected="1초 미만")
result


## 다음 단계

압축 fit 전에 training development/calibration coverage를 별도로 확인합니다. 논문용 전체 run은 training과 공식 test를 모두 포함하고 `full_protocol_complete=True`, `failed=0`, `detector_inference=False`여야 합니다. aligned bundle 또는 recognition model을 바꾸면 같은 run에 섞지 않습니다.
